### Creación del catalog, schemas y volume

In [0]:
-- Creación del catalogo
CREATE CATALOG IF NOT EXISTS electrocasa;

-- Creación de esquemas
CREATE SCHEMA IF NOT EXISTS electrocasa.bronze;
CREATE SCHEMA IF NOT EXISTS electrocasa.silver;
CREATE SCHEMA IF NOT EXISTS electrocasa.gold;
CREATE SCHEMA IF NOT EXISTS electrocasa.monitoring;

-- Creación del volumen landing
CREATE VOLUME IF NOT EXISTS electrocasa.bronze.landing;

### Creación de grupos

Los grupos se deben crear entrando al Account Console -> User management -> Groups -> Add group. Crear siguientes grupos: 
- engineering_team
- analysts_team
- audit_team

Esto para crear los grupos a nivel de la cuenta y puedan se compatibles con Unity Catalog.

# Gobierno y seguridad

### Asignación de permisos

In [0]:
-- Permisos engineering_team
GRANT USE CATALOG ON CATALOG electrocasa TO engineering_team;

-- bronze
GRANT USE SCHEMA ON SCHEMA electrocasa.bronze TO `engineering_team`;
GRANT SELECT ON SCHEMA electrocasa.bronze TO `engineering_team`;
GRANT MODIFY ON SCHEMA electrocasa.bronze TO `engineering_team`;

-- silver
GRANT USE SCHEMA ON SCHEMA electrocasa.silver TO `engineering_team`;
GRANT SELECT ON SCHEMA electrocasa.silver TO `engineering_team`;
GRANT MODIFY ON SCHEMA electrocasa.silver TO `engineering_team`;

-- gold
GRANT USE SCHEMA ON SCHEMA electrocasa.gold TO `engineering_team`;
GRANT SELECT ON SCHEMA electrocasa.gold TO `engineering_team`;
GRANT MODIFY ON SCHEMA electrocasa.gold TO `engineering_team`;

-- monitoring
GRANT USE SCHEMA ON SCHEMA electrocasa.monitoring TO `engineering_team`;
GRANT SELECT ON SCHEMA electrocasa.monitoring TO `engineering_team`;
GRANT MODIFY ON SCHEMA electrocasa.monitoring TO `engineering_team`;

In [0]:
-- Permisos analysts_team
GRANT USE CATALOG ON CATALOG electrocasa TO `analysts_team`;

-- gold
GRANT USE SCHEMA ON SCHEMA electrocasa.gold TO `analysts_team`;
GRANT SELECT ON SCHEMA electrocasa.gold TO `analysts_team`;

In [0]:
-- Permisos audit_team
GRANT USE CATALOG ON CATALOG electrocasa TO `audit_team`;

-- gold
GRANT USE SCHEMA ON SCHEMA electrocasa.gold TO `audit_team`;
GRANT SELECT ON SCHEMA electrocasa.gold TO `audit_team`;

-- auditoria sobre el catalogo
GRANT BROWSE ON CATALOG electrocasa TO `audit_team`;

-- monitoring
GRANT USE SCHEMA ON SCHEMA electrocasa.monitoring TO `audit_team`;
GRANT SELECT ON SCHEMA electrocasa.monitoring TO `audit_team`;

### Emnascaramiento

In [0]:
CREATE SCHEMA IF NOT EXISTS electrocasa.security;

CREATE OR REPLACE FUNCTION electrocasa.security.mask_dni(
    dni STRING
)
RETURNS STRING
RETURN
    CASE
        WHEN is_account_group_member('engineering_team')
            THEN dni
        WHEN dni IS NULL
            THEN NULL
        ELSE '********'
    END;

In [0]:
CREATE OR REPLACE FUNCTION electrocasa.security.mask_salario(
    salario DECIMAL(18,2)
)
RETURNS DECIMAL(18,2)
RETURN
    CASE
        WHEN is_account_group_member('engineering_team')
            THEN salario
        WHEN salario IS NULL
            THEN NULL
        ELSE CAST(0 AS DECIMAL(18,2))
    END;

In [0]:
-- sobre bronze
ALTER TABLE electrocasa.bronze.empleados_bronze
ALTER COLUMN dni
SET MASK electrocasa.security.mask_dni;

ALTER TABLE electrocasa.bronze.empleados_bronze
ALTER COLUMN salario
SET MASK electrocasa.security.mask_salario;

In [0]:
-- Sobre silver
ALTER TABLE electrocasa.silver.empleados_silver
ALTER COLUMN dni
SET MASK electrocasa.security.mask_dni;

ALTER TABLE electrocasa.silver.empleados_silver
ALTER COLUMN salario
SET MASK electrocasa.security.mask_salario;

# Limpiar

In [0]:
-- Remover permisos de engineering_team
REVOKE USE ON CATALOG electrocasa FROM engineering_team;

-- bronze
REVOKE USE ON SCHEMA electrocasa.bronze FROM `engineering_team`;
REVOKE SELECT ON SCHEMA electrocasa.bronze FROM `engineering_team`;
REVOKE MODIFY ON SCHEMA electrocasa.bronze FROM `engineering_team`;

-- silver
REVOKE USE ON SCHEMA electrocasa.silver FROM `engineering_team`;
REVOKE SELECT ON SCHEMA electrocasa.silver FROM `engineering_team`;
REVOKE MODIFY ON SCHEMA electrocasa.silver FROM `engineering_team`;

-- gold
REVOKE USE ON SCHEMA electrocasa.gold FROM `engineering_team`;
REVOKE SELECT ON SCHEMA electrocasa.gold FROM `engineering_team`;
REVOKE MODIFY ON SCHEMA electrocasa.gold FROM `engineering_team`;

-- monitoring
REVOKE USE SCHEMA ON SCHEMA electrocasa.monitoring FROM `engineering_team`;
REVOKE SELECT ON SCHEMA electrocasa.monitoring FROM `engineering_team`;
REVOKE MODIFY ON SCHEMA electrocasa.monitoring FROM `engineering_team`;

In [0]:
-- Permisos analysts_team
REVOKE USE ON CATALOG electrocasa FROM `analysts_team`;

-- gold
REVOKE USE ON SCHEMA electrocasa.gold FROM `analysts_team`;
REVOKE SELECT ON SCHEMA electrocasa.gold FROM `analysts_team`;

In [0]:
-- Permisos audit_team
REVOKE USE ON CATALOG electrocasa FROM `audit_team`;

-- gold
REVOKE USE ON SCHEMA electrocasa.gold FROM `audit_team`;
REVOKE SELECT ON SCHEMA electrocasa.gold FROM `audit_team`;

-- auditoria sobre el catalogo
REVOKE BROWSE ON CATALOG electrocasa FROM `audit_team`;

-- monitoring
REVOKE USE SCHEMA ON SCHEMA electrocasa.monitoring FROM `audit_team`;
REVOKE SELECT ON SCHEMA electrocasa.monitoring FROM `audit_team`;